## Tests — Chat Coach (Grounded Context + Memory)

Validates deterministic coach logic in `model/coach_core.py` without calling the OpenAI API.

## Imports

In [16]:
from __future__ import annotations

import sys
import tempfile
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").is_dir() and (candidate / "artifacts").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing data/ and artifacts/")


ROOT = find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from model.coach_core import (
    affordability_reply,
    answer_question,
    build_grounded_context,
    extract_first_amount_usd,
    load_session,
    write_session,
)


## Amount extraction

In [17]:
assert extract_first_amount_usd("Can I afford $200 today?") == 200.0
assert extract_first_amount_usd("spend 1,234.56?") == 1234.56
assert extract_first_amount_usd("no money mentioned") is None


## Grounded context + deterministic affordability

In [18]:
ctx = build_grounded_context(
    client_id=1696,
    as_of_date="2019-10-31",
    mtd_spend_usd=2919.76,
    limit_usd=1777.00,
    utilization_pct=1.643,
    projected_month_end_usd=2919.76,
    projected_utilization_pct=1.643,
)
assert round(ctx["remaining_budget_usd"], 2) == -1142.76
msg = affordability_reply(ctx, purchase_usd=200.0)
assert "No." in msg


## Coach answer selection (deterministic vs LLM)

In [19]:
r1 = answer_question(context=ctx, question="Can I afford $200 today?", llm_call=None, history=[])
assert r1.used_llm is False
assert "remaining" in r1.content.lower()

def llm_stub(messages):
    assert messages[0]["role"] == "system"
    return "Stubbed LLM reply."

r2 = answer_question(context=ctx, question="What should I do next?", llm_call=llm_stub, history=[])
assert r2.used_llm is True
assert r2.content == "Stubbed LLM reply."


## Amount-only followup uses chat memory

In [20]:
history = [
    {
        "role": "assistant",
        "content": "Tell me the purchase amount (for example: Can I afford a $250 purchase today?).",
    }
]
r3 = answer_question(context=ctx, question="$200", llm_call=None, history=history)
assert r3.used_llm is False
assert "remaining" in r3.content.lower()


## Session persistence

In [21]:
with tempfile.TemporaryDirectory() as d:
    root = Path(d)
    (root / "artifacts").mkdir(parents=True, exist_ok=True)
    msgs = [{"role": "user", "content": "hi"}, {"role": "assistant", "content": "hello"}]
    write_session(root, client_id=1696, messages=msgs)
    loaded = load_session(root, client_id=1696)
    assert loaded == msgs

print("Coach tests: PASS")


Coach tests: PASS


In [22]:
import os

from dotenv import load_dotenv

from model.coach_core import make_openai_chat_call

# override=True: empty shell LLM_API_KEY="" must not block the real .env value
load_dotenv(ROOT / ".env", override=True)

API_KEY = (os.environ.get("LLM_API_KEY") or os.environ.get("OPENAI_API_KEY") or "").strip()
LLM_MODEL = os.environ.get("LLM_MODEL", "gpt-4o-mini")
BASE_URL = os.environ.get("OPENAI_BASE_URL") or os.environ.get("LLM_API_BASE")

assert API_KEY, (
    f"Missing LLM_API_KEY / OPENAI_API_KEY after loading {(ROOT / '.env')}. "
    "Re-run the Imports cell first, then confirm .env has a non-empty key."
)
print("Model:", LLM_MODEL)
print("Base URL:", BASE_URL or "(default OpenAI)")
print("API key loaded:", True, f"(len={len(API_KEY)})")

coach_llm = make_openai_chat_call(model=LLM_MODEL)
reply = coach_llm([{"role": "user", "content": "hello"}])
assert isinstance(reply, str) and reply.strip()
print("Coach OpenAI connection: PASS")
print("Reply:", repr(reply))


Model: gpt-4o-mini
Base URL: https://aibe.mygreatlearning.com/openai/v1
API key loaded: True (len=67)


RuntimeError: LLM request failed: Error code: 429 - {'reason': {'error': 'You exceeded your current quota!!'}}